In [11]:
from pathlib import Path
from typing import Callable
import re, json, ast
import math

In [12]:
# ipynb file in current directory
filename = Path('.').joinpath('assignment-stu.ipynb')
# filename = Path('.').joinpath('assignment.ipynb')

assert filename.exists(), 'indicated file does not exist'

In [13]:
# utility functions
# execute this cell to load functions into memory

def clean_cell(content: list) -> list[str]:
    '''
    content: the cell content to be cleaned, as a list of strings
    '''
    # remove comments
    content = list(map(lambda x: re.sub(r'#.*', '', x), content))

    # remove empty contents
    content = list(filter(lambda x: x.strip() != '' and not x.strip().startswith('pass'), content))

    return content

def get_code(filename: str | Path, question: str | int, data: dict | None = None) -> tuple[str | None, dict]:
    '''
    Retrieves the code for a specific function from a Jupyter notebook JSON file.

    This function reads a Jupyter notebook file (in JSON format) and searches for
    a code cell that defines a function with a name matching the 'question' parameter.
    It returns the cleaned-up code for that function and the notebook's data.

    Args:
        filename: The path to the Jupyter notebook file. It can be a string or a Path object.
        question: The name of the function to search for.
        data: Optional pre-loaded data from the notebook file. If not provided,
              the function will read the file.

    Returns:
        A tuple containing:
        - The code of the target function as a single string, or None if not found.
        - The dictionary of the notebook's JSON data.
    
    Raises:
        FileNotFoundError: If the specified file does not exist.
    '''
    code: str = ''
    try:
        if data is None:
            with open(filename, 'r') as f:
                data = json.load(f)
   
        target_fn: str = f'def {question}'
    
        for cell in data.get('cells', []):
            if cell.get('cell_type') == 'code':
                source = cell.get('source', [])
                # identify target function
                if any(i.startswith(target_fn) for i in source):  
                    code = ''.join(clean_cell(source))
                    
        return code, data
    except FileNotFoundError as e:
        print(f'{e}')
    return code, data

def get_function(filename: str | Path, question: str | int, data: dict | None = None) -> tuple[Callable | None, dict]:
    '''
    Uses `get_code` to extract relevant question code from given filename.
    data = Optional pre-loaded json dictionary data from the notebook file.
    '''

    if isinstance(question, int): question = f'q{question}'
    
    code, data = get_code(filename, question, data)

    func = None
    if code:
        try:
            # run the code into a fresh environment
            env = {}
            exec(code, env)
            # get the function name from the code
            func = env.get(question)
        except Exception as e:
            print(f'[0] Error in code execution: {e}')
    return func, data

def do_tests(test_cases: dict, func: Callable, total_score: float = 0, total_comments: str = '', max_score: float = 0, Q6 = False) -> tuple[float, str, float]:
    for test_case, details in test_cases.items():
        score = details['score']
        max_score += sum(score) if Q6 else score
        try:
            # Construct fresh objects inside try so constructor errors fail only this test.
            inputs = details['input_factory']() if 'input_factory' in details else details['input']
            result = func(*inputs, **details.get("kwargs", {}))
            if Q6:
                if result[0] == details['output'][0]:
                    total_score += score[0]
                    total_comments += f'[{score[0]}] {test_case} feasibility check passed.\n'
                else:
                    total_comments += f'[0] {test_case} feasibility check failed.\n'

                if result[1] == details['output'][1]:
                    total_score += score[1]
                    total_comments += f'[{score[1]}] {test_case} passed.\n'
                else:
                    total_comments += f'[0] {test_case} failed.\n'
            else:
                if (isinstance(result, (int, float)) and math.isclose(result, details['output'])) or result == details['output']:
                    total_score += score
                    total_comments += f'[{score}] {test_case} passed.\n'
                else:
                    total_comments += f'[0] {test_case} failed.\n'

        except Exception:
            total_comments += f'[0] {test_case} failed due to error in execution.\n'
    return total_score, total_comments, max_score

### Testing starts here...

In [14]:
# search for your q1 solution in the file
func, data = get_function(filename, 1)

if func:
    # test the code with test cases
    sample_test_cases = {
        "sample_test_case_1": {
            "input": [[("mug", 9), ("lamp", 45), ("desk", 120)]],
            "output": ("mug", "desk"),
            "score": 0.5
        },
        "sample_test_case_2": {
            "input": [[("pen", 4)]],
            "output": ("pen", "pen"),
            "score": 0.5
        }
    }

    open_test_cases = {
        "open_test_case_1": {
            "input": [[("tape", 3), ("pen", 4), ("clip", 4)]],
            "output": ("tape", "pen"),
            "score": 1
        }
    }

    total_score, total_comments, max_score = do_tests(sample_test_cases, func)
    total_score, total_comments, max_score = do_tests(open_test_cases, func, total_score, total_comments, max_score)

    print(f'Total Score: {total_score:.1f}/{max_score:.1f}')
    print(total_comments)
else:
    print('[0] No valid code found for q1. Please check your notebook.')

Total Score: 2.0/2.0
[0.5] sample_test_case_1 passed.
[0.5] sample_test_case_2 passed.
[1] open_test_case_1 passed.



In [15]:
# search for your q2 solution in the file
func = get_function(filename, 2, data)[0]

if func:
    # test the code with test cases
    sample_test_cases = {
        "sample_test_case_1": {
            "input": [[("Shape of You", 240), ("Havana", 217)]],
            "output": ([True, True], 2, 457, "Shape of You"),
            "score": 0.5
        },
        "sample_test_case_2": {
            "input": [[("Lofi Rain", 180), ("Lofi Rain", 200)]],
            "output": ([True, False], 1, 180, "Lofi Rain"),
            "score": 0.5
        }
    }

    open_test_cases = {
        "open_test_case_1": {
            "input": [[("Eye of the Tiger", 245), ("Stronger", 312), ("Titanium", 245)]],
            "output": ([True, True, True], 3, 802, "Stronger"),
            "score": 1
        }
    }

    total_score, total_comments, max_score = do_tests(sample_test_cases, func)
    total_score, total_comments, max_score = do_tests(open_test_cases, func, total_score, total_comments, max_score)

    print(f'Total Score: {total_score:.1f}/{max_score:.1f}')
    print(total_comments)
else:
    print('[0] No valid code found for q2. Please check your notebook.')


Total Score: 2.0/2.0
[0.5] sample_test_case_1 passed.
[0.5] sample_test_case_2 passed.
[1] open_test_case_1 passed.



In [16]:
# search for your q3 solution in the file
func = get_function(filename, 3, data)[0]

if func:
    # test the code with test cases
    sample_test_cases = {
        "sample_test_case_1": {
            "input": [[("regular", "Ana", 30)]],
            "output": ([30], ["Ana pays $30"], 30),
            "score": 0.5
        },
        "sample_test_case_2": {
            "input": [[("child", "Ben", 25)]],
            "output": ([12], ["Ben pays $12"], 12),
            "score": 0.5
        }
    }

    open_test_cases = {
        "open_test_case_1": {
            "input": [[("child", "Ben", 25), ("senior", "Chen", 40)]],
            "output": ([12, 28], ['Ben pays $12', 'Chen pays $28'], 40),
            "score": 1
        }
    }

    total_score, total_comments, max_score = do_tests(sample_test_cases, func)
    total_score, total_comments, max_score = do_tests(open_test_cases, func, total_score, total_comments, max_score)

    print(f'Total Score: {total_score:.1f}/{max_score:.1f}')
    print(total_comments)
else:
    print('[0] No valid code found for q3. Please check your notebook.')


Total Score: 2.0/2.0
[0.5] sample_test_case_1 passed.
[0.5] sample_test_case_2 passed.
[1] open_test_case_1 passed.



In [17]:
# search for your q4 solution in the file
func = get_function(filename, 4, data)[0]

if func:
    # test the code with test cases
    sample_test_cases = {
        "sample_test_case_1": {
            "input": [{
                "Toy Story 3": {"year": 2010, "runtime": 103, "budget": 200.0, "revenue": 1068.9},
                "Toy Story 4": {"year": 2019, "runtime": 100, "budget": 200.0, "revenue": 1073.8}
            }],
            "output": ['Toy Story 3', 'Toy Story 4'],
            "score": 0.5
        },
        "sample_test_case_2": {
            "input": [{
                "Encanto": {"year": 2021, "runtime": 102, "budget": 150.0, "revenue": 256.8}, 
                "The Little Mermaid": {"year": 2023, "runtime": 135, "budget": 250.0, "revenue": 569.6}
            }],
            "output": [],
            "score": 0.5
        }
    }

    open_test_cases = {
        "open_test_case_1": {
            "input": [{
                "Shrek": {"year": 2001, "runtime": 90, "budget": 60.0, "revenue": 489.7},
                "Shrek 2": {"year": 2004, "runtime": 93, "budget": 150.0, "revenue": 932.5},
                "Shrek the Third": {"year": 2007, "runtime": 93, "budget": 160.0, "revenue": 808.3},
                "Shrek Forever After": {"year": 2010, "runtime": 93, "budget": 165.0, "revenue": 752.6}
            }],
            "output": ['Shrek', 'Shrek 2', 'Shrek the Third'],
            "score": 1
        }
    }

    total_score, total_comments, max_score = do_tests(sample_test_cases, func)
    total_score, total_comments, max_score = do_tests(open_test_cases, func, total_score, total_comments, max_score)

    print(f'Total Score: {total_score:.1f}/{max_score:.1f}')
    print(total_comments)
else:
    print('[0] No valid code found for q4. Please check your notebook.')


Total Score: 2.0/2.0
[0.5] sample_test_case_1 passed.
[0.5] sample_test_case_2 passed.
[1] open_test_case_1 passed.



In [18]:
# search for your q5 solution in the file
func = get_function(filename, 5, data)[0]

if func:
    # test the code with test cases
    sample_test_cases = {
        "sample_test_case_1": {
            "input": ["Americano"],
            "kwargs": {"coffee": 60, "water": 180},
            "output": ('Americano', 240, False),
            "score": 0.5
        },
        "sample_test_case_2": {
            "input": ["Cappuccino"],
            "kwargs": {"milk": 60, "coffee": 60, "milk_froth": 60}, 
            "output": ('Cappuccino', 180, True),
            "score": 0.5
        }
    }

    open_test_cases = {
        "open_test_case_1": {
            "input": ["Latte"],
            "kwargs": {"milk": 200, "coffee": 60},
            "output": ('Latte', 260, True),
            "score": 1
        }
    }

    total_score, total_comments, max_score = do_tests(sample_test_cases, func)
    total_score, total_comments, max_score = do_tests(open_test_cases, func, total_score, total_comments, max_score)

    print(f'Total Score: {total_score:.1f}/{max_score:.1f}')
    print(total_comments)
else:
    print('[0] No valid code found for q5. Please check your notebook.')


Total Score: 2.0/2.0
[0.5] sample_test_case_1 passed.
[0.5] sample_test_case_2 passed.
[1] open_test_case_1 passed.



In [19]:
# Provided classes
class Soldier:
    def __init__(self):
        self.health = 50
        self.attack = 20
        self.defense = 10

class Equipment:
    def __init__(self, bonus):
        self.bonus_health = bonus[0]
        self.bonus_attack = bonus[1]
        self.bonus_defense = bonus[2]


class Hero(Soldier):
    def __init__(self, list_of_equipments=None):
        super().__init__()
        if list_of_equipments:
            for Eqm in list_of_equipments:
                self.health += Eqm.bonus_health
                self.attack += Eqm.bonus_attack
                self.defense += Eqm.bonus_defense


# search for your q6 solution in the file
func = get_function(filename, 6, data)[0]

if func:
    # Build fresh fighters using the fixed instructor-provided classes.
    sample_test_cases = {
        "sample_test_case_1": {
            "input_factory": lambda: [
                Soldier(),
                Hero([Equipment([0, 10, 3])])  # sword
            ],
            "output": 2,
            "score": 0.5
        },
        "sample_test_case_2": {
            "input_factory": lambda: [
                Hero([
                    Equipment([0, 10, 3]),  # sword
                    Equipment([5, 0, 9])    # shield
                ]),
                Hero([Equipment([0, 14, 7])])  # spear
            ],
            "output": 1,
            "score": 0.5
        }
    }

    open_test_cases = {
        "open_test_case_1": {
            "input_factory": lambda: [
                Hero([Equipment([0, 999, 0])]),
                Hero([
                    Equipment([0, 999, 0]),  # thayes_gauntlet
                    Equipment([0, 0, 100])   # big_shield
                ])
            ],
            "output": 0,
            "score": 1
        }
    }

    # q6 returns one integer, so use ordinary scoring (Q6=False).
    total_score, total_comments, max_score = do_tests(sample_test_cases, func)
    total_score, total_comments, max_score = do_tests(open_test_cases, func, total_score, total_comments, max_score)

    print(f'Total Score: {total_score:.1f}/{max_score:.1f}')
    print(total_comments)
else:
    print('[0] No valid code found for q6. Please check your notebook.')


Total Score: 2.0/2.0
[0.5] sample_test_case_1 passed.
[0.5] sample_test_case_2 passed.
[1] open_test_case_1 passed.

